In [ ]:
import numpy as np
from scipy.sparse.linalg import svds

In [ ]:
dataset = [
    {
        "filename": "story938.txt",
        "text": "Anger explodes in streets of Athens. The riot, widely described as one of the worst in Athens since Greece began dealing with its crippling debt crisis in 2010...",
        "words": [
            "explodes", "street", "riot", "widely", "describe", "one", "bad", "since", 
            "begin", "deal", "cripple", "debt", "crisis", "break", "approve", "new", 
            "package", "austerity", "measure", "return", "new", "eurozone", "bailout", 
            "country", "among", "crowd", "estimate", "police", "hurl", "rock", "toward", 
            "police", "respond", "tear", "gas", "least", "people", "arrested", "police", 
            "say", "say", "investigator", "still", "tally", "property", "damage", "say", 
            "building", "damage", "fire", "numerous", "others", "loot", "damaged", 
            "building", "bank", "cafes", "movie", "theater", "news", "agency", "report", 
            "violent", "protest", "reflect", "increase", "angst", "crush", "economic", 
            "problem", "say", "protest", "feel", "end", "financial", "suffer", "people", 
            "every", "day", "get", "poor", "become", "homeless", "especially", "say", 
            "scatter", "protester", "tear", "gas", "package", "include", "deep", "cut", 
            "government", "spending", "wage", "pension", "help", "pave", "way", 
            "eurozone", "finance", "minister", "sign", "new", "billion", "billion", 
            "bailout", "deal", "pass", "vote", "need", "fund", "order", "meet", 
            "billion", "debt", "repayment", "due", "next", "month", "vote", "call", 
            "calm", "vandalism", "violence", "place", "democracy", "tolerate", "told", 
            "lawmaker", "urge", "approve", "deal", "critical", "time", "luxury", 
            "conflict", "say", "comment", "make", "speech", "even", "warn", "lawmaker", 
            "government", "would", "able", "pay", "salary", "cover", "service", 
            "import", "basic", "good", "plan", "approve", "spoke", "possibility", 
            "catastrophic", "bankruptcy", "hard", "program", "tough", "program", 
            "also", "envisage", "painful", "sacrifice", "say", "add", "plan", "would", 
            "put", "back", "stable", "ground", "criticize", "program", "demonstrate", 
            "day", "eurozone", "figure", "stack", "isolated", "break", "protester", 
            "rally", "front", "building", "mood", "calmer", "day", "earlier", "protest", 
            "disperse", "youth", "smash", "pavement", "begin", "throw", "stone", "piece", 
            "marble", "well", "cocktails", "police", "respond", "stun", "grenade", 
            "protest", "widely", "describe", "bad", "since", "debt", "crisis", "begin", 
            "year", "three", "people", "die", "fire", "bomb", "hit", "bank", "central", 
            "protest", "austerity", "measure", "though", "sweeping", "reform", "package", 
            "agree", "troika", "make", "approved", "lawmaker", "must", "still", "prime", 
            "minister", "head", "bring", "together", "finance", "minister", "say", 
            "assurance", "also", "need", "bailout", "could", "pay", "political", 
            "leader", "must", "pledge", "continue", "implement", "measure", "upcoming", 
            "election", "say", "must", "also", "find", "million", "structural", 
            "expenditure", "cut", "add", "bailout", "deal", "would", "result", 
            "significant", "loss", "bondholder", "intend", "help", "reduce", "debt", 
            "gross", "domestic", "product", "currently", "owe", "billion", "come", 
            "close", "default", "nation", "struggle", "follow", "austerity", "measure", 
            "economic", "reform", "condition", "bailout", "package", "time", "economy", 
            "recession", "year", "many", "analyst", "warn", "additional", "austerity", 
            "could", "make", "situation", "bad", "stock", "higher", "morning", "follow", 
            "passage", "austerity", "package", "market", "anxious", "potential", 
            "impact", "default", "world", "economy"
        ]
    },
    # Secondary document to establish contextual cross-document matrix dimension (n_docs >= 2)
    {
        "filename": "tech_sample.txt",
        "text": "Machine learning algorithms and data models revolutionize statistical computing.",
        "words": [
            "machine", "learning", "data", "model", "algorithm", "data", "statistical",
            "computing", "system", "analyst", "economic", "market", "finance"
        ]
    }
]




# ==============================================================================
# Step 1: Document-Term Matrix Construction & Log-Entropy Normalization
# ==============================================================================

def makeDocumentTermMatrix(data):
    """
    Constructs a 2D NumPy array of size (Words x Documents).
    Matrix cell [i, j] stores the raw count of word i in document j.
    """
    vocabulary = sorted(list({w for doc in data for w in doc["words"]}))
    word_to_idx = {w: i for i, w in enumerate(vocabulary)}
    
    num_words = len(vocabulary)
    num_docs = len(data)
    
    A = np.zeros((num_words, num_docs), dtype=float)
    for j, doc in enumerate(data):
        for word in doc["words"]:
            i = word_to_idx[word]
            A[i, j] += 1.0
            
    return A, vocabulary, word_to_idx


def logEntropyNormalization(A):
    """
    Applies standard Log-Entropy scaling:
      Local Factor:   L_ij = log(tf_ij + 1)
      Global Factor:  G_i  = 1 + (1 / log(N)) * sum_j (p_ij * log(p_ij))
                      where p_ij = tf_ij / gf_i, and gf_i = sum_j tf_ij
      Scaled Matrix:  S_ij = L_ij * G_i
    """
    num_words, num_docs = A.shape
    
    # 1. Local Factor
    L = np.log(A + 1.0)
    
    if num_docs <= 1:
        return L
    
    # 2. Global Factor
    row_sums = np.sum(A, axis=1, keepdims=True)
    p = np.divide(A, row_sums, out=np.zeros_like(A), where=(row_sums > 0))
    
    log_p = np.zeros_like(p)
    nonzero_mask = p > 0
    log_p[nonzero_mask] = np.log(p[nonzero_mask])
    
    entropy = np.sum(p * log_p, axis=1) / np.log(num_docs)
    G = 1.0 + entropy  # Shape: (num_words,)
    
    # 3. Scaled Dataset Matrix
    S = L * G[:, np.newaxis]
    return S


A, vocab, word_to_idx = makeDocumentTermMatrix(dataset)
S = logEntropyNormalization(A)

print(f"Matrix shape: {S.shape[0]} terms x {S.shape[1]} documents")

# ==============================================================================
# Step 2: Dimensionality Reduction via Truncated SVD
# ==============================================================================

k = 2

# Truncated SVD decomposition down to k=2 latent concepts
U, sigma, Vt = np.linalg.svd(S, full_matrices=False)
U_k = U[:, :k]
sigma_k = sigma[:k]
Vt_k = Vt[:k, :]

# Project words into dense low-dimensional concept space: W_proj = S · (V_k)^T
# Equivalence check: S · V_k^T = U_k · Σ_k
projected_words = np.dot(S, Vt_k.T)

print(f"Projected Words coordinate matrix shape: {projected_words.shape}")

# ==============================================================================
# Step 3: Latent Similarity Mapping
# ==============================================================================

def find_closest_words(target_word, top_n=3):
    """
    Finds the top N closest terms to target_word using Cosine Similarity
    over the low-dimensional projected concept vectors.
    """
    if target_word not in word_to_idx:
        raise ValueError(f"Word '{target_word}' not found in vocabulary.")
        
    target_idx = word_to_idx[target_word]
    target_vec = projected_words[target_idx]
    
    target_norm = np.linalg.norm(target_vec)
    if target_norm == 0:
        return []
        
    norms = np.linalg.norm(projected_words, axis=1)
    # Cosine Similarity: (A · B) / (||A|| * ||B||)
    similarities = np.dot(projected_words, target_vec) / (norms * target_norm + 1e-12)
    
    ranked_indices = np.argsort(-similarities)
    
    results = []
    for idx in ranked_indices:
        if idx != target_idx:
            results.append((vocab[idx], similarities[idx]))
        if len(results) == top_n:
            break
            
    return results


# Output Evaluation for target word "data"
target = "data"
top_matches = find_closest_words(target, top_n=3)

print(f"\nTop 3 contextually closest words to '{target}':")
for rank, (word, score) in enumerate(top_matches, start=1):
    print(f"{rank}. {word} (Cosine Similarity: {score:.4f})")